# Differentiable reachability: PGD over the nominal initial state

This notebook uses reachability analysis inside an optimization loop. We choose
a nominal initial state `x_nom` from a box, define the input set as
`x_nom +/- perturb_radius`, and minimize the reachable tube size with projected
gradient descent (PGD).

In [1]:
from pathlib import Path

repo_root = Path.cwd()

import jax
# CPU usually compiles faster and is convenient for a tutorial.
# Comment out the next line to run on GPU.
jax.config.update("jax_platform_name", "cpu")

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")

import jax.numpy as jnp

from src.reachability import CT_Ctl_Reach
from src.utils.load import load_analytic_dynamics, load_controller

## 1. Small cartpole reachability problem

We optimize only the four state coordinates of the nominal point. The fifth
augmented coordinate, `u1`, is initialized at zero and then imposed by the
controller during reachability analysis.

In [2]:
state_dim = 4
action_dim = 1
variable_names = ["x1", "x2", "x3", "x4"]

# Keep this small for a tutorial. The first gradient call includes JIT compile.
n_total_steps = 10
n_steps_per_control = 4
step_size = 0.005

dynamics_path = repo_root / "models/dynamics/ct_ctl/cartpole.py"
controller_path = repo_root / "models/controllers/ct_ctl/cartpole.onnx"

rhs = load_analytic_dynamics(
    str(dynamics_path),
    D_s=state_dim,
    D_a=action_dim,
    discrete=False,
)
controller = load_controller(str(controller_path))

reach = CT_Ctl_Reach(
    rhs=rhs,
    state_dim=state_dim,
    action_dim=action_dim,
    nn_dyn=False,
    controller=controller,
    n_steps_per_control=n_steps_per_control,
    step_size=step_size,
    init_remainder=jnp.array([1e-5]),
    frr_rounds=10,
    frr_stop_ratio=0.95,
    sr_window_size=1000,
    reference_dim=0,
)

## 2. Nominal range and perturbation radius

For each candidate nominal point, the verified input set is a single augmented
box. There is no input splitting in this notebook.

In [3]:
nominal_lo = jnp.array([-0.040, -0.020, -0.010, -0.010], dtype=jnp.float32)
nominal_hi = jnp.array([-0.020,  0.000,  0.010,  0.010], dtype=jnp.float32)
perturb_radius = jnp.array([0.002, 0.001, 0.001, 0.001], dtype=jnp.float32)

def make_initial_box(x_nom):
    state_lo = x_nom - perturb_radius
    state_hi = x_nom + perturb_radius
    action0 = jnp.array([0.0], dtype=x_nom.dtype)
    z0_lo = jnp.concatenate([state_lo, action0])[None, :]
    z0_hi = jnp.concatenate([state_hi, action0])[None, :]
    return z0_lo, z0_hi

## 3. Differentiable tube-size objective

`tube_size(x_nom)` runs reachability and returns the average normalized width of
the reachable state tube. Because the reachability code is written in JAX, we
can differentiate this scalar objective with respect to `x_nom`.

In [4]:
def tube_size(x_nom):
    z0_lo, z0_hi = make_initial_box(x_nom)
    _, lowers, uppers, _, _ = reach.verify(z0_lo, z0_hi, n_total_steps)

    # Exclude t=0 so the objective reflects reachable growth, not just the
    # fixed input perturbation box.
    widths = jnp.maximum(uppers[:, 1:, :state_dim] - lowers[:, 1:, :state_dim], 0.0)
    normalized_widths = widths / (2.0 * perturb_radius)
    return jnp.mean(normalized_widths)

tube_value_and_grad = jax.value_and_grad(tube_size)
value_and_grad = jax.jit(tube_value_and_grad)

The first call compiles the objective and its gradient. Later calls with the
same shape are much faster.

In [5]:
x0 = 0.5 * (nominal_lo + nominal_hi)
value, grad = value_and_grad(x0)

print(f"tube_size(x0) = {float(value):.6e}")
for name, g in zip(variable_names, grad):
    print(f"d objective / d {name}: {float(g): .6e}")

tube_size(x0) = 4.088602e+00
d objective / d x1:  5.738971e+01
d objective / d x2:  5.462347e+01
d objective / d x3:  2.044001e+02
d objective / d x4:  3.851910e+01


## 4. Plain single-start PGD

This first version is intentionally simple: it starts from `x0`, uses a Python
loop, and prints the optimized nominal point. It is useful for reading and
debugging the optimization logic.

In [6]:
def project_nominal(x):
    return jnp.clip(x, nominal_lo, nominal_hi)

def pgd_one_start(x_start, *, pgd_steps=4, step_scale=1e-5):
    x = project_nominal(x_start)
    history = []

    for _ in range(pgd_steps):
        value, grad = value_and_grad(x)
        history.append(float(value))
        x = project_nominal(x - step_scale * grad)
        print(f"iteration {_}: tube_size = {float(value):.6e}, x = {x}")

    final_value, _ = value_and_grad(x)
    history.append(float(final_value))
    return x, float(final_value), history

best_x, best_value, best_history = pgd_one_start(x0, pgd_steps=4, step_scale=1e-5)

print(f"initial tube_size = {best_history[0]:.6e}")
print(f"final tube_size   = {best_value:.6e}")
print("\noptimized nominal point:")
for name, xi in zip(variable_names, best_x):
    print(f"{name} = {float(xi): .8e}")
print(f"tube_size = {best_value:.6e}")

iteration 0: tube_size = 4.088602e+00, x = [-0.0305739  -0.01054623 -0.002044   -0.00038519]
iteration 1: tube_size = 3.660740e+00, x = [-0.0309939  -0.01095138 -0.0034879  -0.00065216]
iteration 2: tube_size = 3.408860e+00, x = [-0.03141399 -0.01135706 -0.00498253 -0.00092858]
iteration 3: tube_size = 3.143491e+00, x = [-0.03182523 -0.01175207 -0.00648786 -0.00120934]
initial tube_size = 4.088602e+00
final tube_size   = 2.875422e+00

optimized nominal point:
x1 = -3.18252295e-02
x2 = -1.17520662e-02
x3 = -6.48785522e-03
x4 = -1.20934146e-03
tube_size = 2.875422e+00


## 5. Fully jittable batched PGD

For multi-start searches, avoid calling `pgd_one_start` in a Python loop. This
version keeps all starts in one batch, uses `jax.lax.scan` for PGD iterations,
and can be compiled with `jax.jit`.

In [7]:
def make_initial_boxes(x_nom_batch):
    state_lo = x_nom_batch - perturb_radius[None, :]
    state_hi = x_nom_batch + perturb_radius[None, :]
    action0 = jnp.zeros((x_nom_batch.shape[0], action_dim), dtype=x_nom_batch.dtype)
    z0_lo = jnp.concatenate([state_lo, action0], axis=1)
    z0_hi = jnp.concatenate([state_hi, action0], axis=1)
    return z0_lo, z0_hi

def batched_tube_sizes(x_nom_batch):
    z0_lo, z0_hi = make_initial_boxes(x_nom_batch)
    _, lowers, uppers, _, _ = reach.verify(z0_lo, z0_hi, n_total_steps)
    widths = jnp.maximum(uppers[:, 1:, :state_dim] - lowers[:, 1:, :state_dim], 0.0)
    normalized_widths = widths / (2.0 * perturb_radius)[None, None, :]
    return jnp.mean(normalized_widths, axis=(1, 2))

def batched_tube_loss(x_nom_batch):
    return jnp.sum(batched_tube_sizes(x_nom_batch))

batched_tube_grad = jax.grad(batched_tube_loss)

def pgd_multi_start(starts, *, pgd_steps=4, step_scale=1e-5):
    starts = project_nominal(starts)

    def step(xs, _):
        values = batched_tube_sizes(xs)
        grads = batched_tube_grad(xs)
        xs_next = project_nominal(xs - step_scale * grads)
        return xs_next, values

    final_xs, history = jax.lax.scan(step, starts, None, length=pgd_steps)
    final_values = batched_tube_sizes(final_xs)
    history = jnp.concatenate([history, final_values[None, :]], axis=0)
    best_idx = jnp.argmin(final_values)
    return final_xs, final_values, history, best_idx

batched_pgd = jax.jit(pgd_multi_start, static_argnames=("pgd_steps",))

In [8]:
# Example run:
num_starts = 8
key = jax.random.PRNGKey(0)
starts = jax.random.uniform(
    key,
    shape=(num_starts, state_dim),
    minval=nominal_lo,
    maxval=nominal_hi,
)
final_xs, final_values, history, best_idx = batched_pgd(starts, pgd_steps=4)
best_x = final_xs[best_idx]
best_value = final_values[best_idx]

print(f"final tube_size   = {best_value:.6e}")
print("\noptimized nominal point:")
for name, xi in zip(variable_names, best_x):
    print(f"{name} = {float(xi): .8e}") 

final tube_size   = 2.319636e+00

optimized nominal point:
x1 = -3.57440689e-02
x2 = -9.01435345e-03
x3 = -9.99999978e-03
x4 = -1.01328103e-03
